# Aula 07 · Arquivos de texto e CSV

Esta aula apresenta o [capítulo 7 do site](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/). A ideia central: **arquivo não é assunto
novo**. Ler um arquivo é um laço que entrega uma linha por vez; o CSV entrega um
dicionário por vez. Daí em diante, são os padrões da Unidade 1 sobre a lista de
dicionários da Unidade 2 — com duas linhas de defesa no começo de todo laço.

**Ao fim da aula você consegue:**

1. ler um arquivo linha a linha com `with open(...)` e se defender da linha em
   branco do fim;
2. escrever um arquivo, sabendo a diferença entre `"w"` e `"a"`;
3. ler um CSV com `csv.DictReader`, guardar os registros numa lista e calcular com
   eles — convertendo o texto em número.

**Roteiro:** 🔥 aquecimento · 📟 chamado · 1. abrir, ler, fechar · 2. limpando e
contando · 3. escrever · 4. CSV · 5. guardando as linhas · 6. o CSV vira o que você
já sabe · 7. quando o arquivo não está lá · 📟 resolvendo o chamado · 🚪 antes de
sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

A célula ⚙️ desta aula também **cria os arquivos de exemplo** na pasta `trabalho/`
(veja no ícone de pasta, à esquerda do Colab). Eles **somem quando o Colab
reinicia** — se aparecer `FileNotFoundError`, rode a célula ⚙️ de novo.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
from pathlib import Path

PASTA = Path("trabalho")
PASTA.mkdir(exist_ok=True)
(PASTA / "alarmes.log").write_text(
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3\n"
    "2026-03-02 09:00:00 INFO SWITCH-NORTE-02 porta ativada\n"
    "2026-03-02 09:41:12 CRITICAL ONU-SUL-4512 sem resposta ha 12 minutos\n"
    "2026-03-02 10:02:55 CRITICAL OLT-CENTRO-01 temperatura acima do limite\n"
    "\n",
    encoding="utf-8",
)
(PASTA / "medicoes.csv").write_text(
    "equipamento,potencia_dbm,estado\n"
    "OLT-CENTRO-01,-21.4,UP\n"
    "ONU-SUL-4512,-27.0,UP\n"
    "OLT-NORTE-02,-19.8,DOWN\n",
    encoding="utf-8",
)
(PASTA / "estados_do_dia.csv").write_text(
    "equipamento,hora,estado\n"
    "OLT-CENTRO-01,00,UP\nONU-SUL-4512,00,UP\nOLT-NORTE-02,00,UP\n"
    "OLT-CENTRO-01,01,UP\nONU-SUL-4512,01,DOWN\nOLT-NORTE-02,01,UP\n"
    "OLT-CENTRO-01,02,DOWN\nONU-SUL-4512,02,DOWN\nOLT-NORTE-02,02,UP\n"
    "OLT-CENTRO-01,03,UP\nONU-SUL-4512,03,UP\nOLT-NORTE-02,03,UP\n"
    "OLT-CENTRO-01,04,UP\nONU-SUL-4512,04,UP\nOLT-NORTE-02,04,UP\n"
    "OLT-CENTRO-01,05,UP\nONU-SUL-4512,05,UP\nOLT-NORTE-02,05,UP\n",
    encoding="utf-8",
)
print("arquivos de exemplo criados em", PASTA.name)

## 🔥 Aquecimento — da aula passada

Sem rodar nada: o que este trecho imprime?

```python
ontem = {"OLT-A", "ONU-B", "RADIO-C"}
hoje = {"OLT-A", "RADIO-C", "SWITCH-D"}
print(sorted(ontem - hoje))
print(len(hoje | ontem))
```

<details>
<summary><b>Resposta</b></summary>

Imprime `['ONU-B']` (o que estava ontem e sumiu hoje) e `4` (todos os já vistos:
OLT-A, ONU-B, RADIO-C e SWITCH-D).

</details>

## 📟 O chamado de hoje

> **Chamado #0718 — NOC Maré Net**
>
> *"Estagiário, o contrato com o cliente corporativo promete **disponibilidade** de
> cada enlace. O coletor grava, de hora em hora, o estado de cada equipamento
> (`UP` ou `DOWN`) num CSV. Preciso da **porcentagem do tempo em que cada um ficou
> no ar** — hoje alguém abre o arquivo na planilha e conta na mão."*

No fim da aula você lê o CSV (`trabalho/estados_do_dia.csv`) e calcula isso.

## 1. Abrir, ler, fechar

`with open(caminho, encoding="utf-8") as arquivo:` abre o arquivo e o **fecha
sozinho** ao sair do bloco. Um `for` sobre o arquivo entrega **uma linha por vez** —
e cada linha vem como ela está no arquivo, com tudo o que tem nela.

📖 [capítulo 7 · Abrir, ler, fechar](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#abrir-ler-fechar)

**✍️ Passo 1.** Abra `"trabalho/alarmes.log"` com `with open(..., encoding="utf-8") as arquivo:` e,
dentro, percorra `for linha in arquivo:` imprimindo `repr(linha)`.

In [ ]:
# ✍️ passo 1

**Preveja:** o arquivo tem 4 alarmes. Quantas linhas o laço vai imprimir? O `repr` mostra algo
que o `print` comum esconderia?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Imprime **5** linhas. Cada uma termina com `\n` — a quebra de linha faz parte da
linha lida. E a última é só `'\n'`: uma **linha em branco** no fim do arquivo. O
`for` a entrega como qualquer outra.

</details>

## 2. Limpando e contando

Aquela linha em branco derruba qualquer código que faça `linha.split()[2]`. Por isso
**todo laço de leitura** deste curso, daqui em diante, abre com duas linhas de
defesa: limpar a linha e pular a vazia.

📖 [capítulo 7 · Limpando e contando](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#limpando-e-contando)

**✍️ Passo 2.** Conte os críticos **sem defesa**: `criticos = 0` e, no laço sobre o arquivo, **se**
`linha.split()[2] == "CRITICAL"`, some 1. Imprima `criticos`.

In [ ]:
# ✍️ passo 2

**Preveja:** funciona? Se não, em qual linha do arquivo quebra?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Quebra com `IndexError` — na **última** volta, a da linha em branco: `"\n".split()`
é uma lista vazia, e não existe `[2]`. Os três críticos foram contados, mas o
programa morreu antes de imprimir.

</details>

**✍️ Passo 3.** Acrescente as duas linhas de defesa no começo do laço: `linha = linha.strip()` e
`if len(linha) == 0: continue`. Rode de novo.

In [ ]:
# ✍️ passo 3

**Preveja:** agora quantas linhas são contadas como críticas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai `3`. O `strip()` tira o `\n` (e espaços) das pontas; a linha em branco vira `""`,
tem tamanho zero, e o `continue` pula para a próxima volta sem executar o resto.

</details>

> ⚠️ **Armadilha.** Achar que "número de alarmes = número de linhas". Todo arquivo de texto bem formado
> termina com uma quebra, e às vezes com uma linha vazia a mais. É a causa número um
> de `IndexError` em script de log. As duas linhas de defesa abrem **todo** laço de
> leitura.

**✍️ Passo 4.** Outra forma, quando o arquivo cabe na memória:
`linhas = arquivo.read().splitlines()` dentro do `with`. Depois do bloco, imprima
`len(linhas)` e `linhas[0].split()[3]`.

In [ ]:
# ✍️ passo 4

**Preveja:** `len(linhas)` dá 4 ou 5?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Dá `5` — a linha vazia continua lá, agora como `""` (o `splitlines` já tira o `\n`).
E `OLT-CENTRO-01`. Repare que `linhas` continua existindo **depois** do `with`: o
arquivo fechou, mas a lista está na memória.

</details>

### 🎯 Sua vez — Contar por severidade

Escreva `conta_severidade(caminho, severidade)`, que abre o arquivo de log e devolve
quantas linhas têm aquela severidade — com as duas linhas de defesa.

In [ ]:
def conta_severidade(caminho, severidade):
    # sua solução aqui
    pass

In [ ]:
confere(conta_severidade, [
    (("trabalho/alarmes.log", "CRITICAL"), 3),
    (("trabalho/alarmes.log", "INFO"), 1),
    (("trabalho/alarmes.log", "WARNING"), 0),
])

<details>
<summary><b>💡 Dica</b></summary>

É o passo anterior dentro de uma função, com `severidade` no lugar de
`"CRITICAL"` e `return` no fim — **fora** do `with` ou dentro, tanto faz, mas fora
do `for`.

</details>

## 3. Escrever

Para escrever, abre-se com um **modo**: `"w"` cria o arquivo ou **apaga** o que já
existia; `"a"` acrescenta no fim. `arquivo.write(texto)` não põe a quebra de linha
sozinho — o `"\n"` é por sua conta.

📖 [capítulo 7 · Escrever](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#escrever)

**✍️ Passo 5.** Abra `"trabalho/relatorio.txt"` com `"w"` e escreva duas linhas:
`arquivo.write("OLT-CENTRO-01     2\n")` e `arquivo.write("ONU-SUL-4512      1\n")`.
Depois do bloco, imprima `Path("trabalho/relatorio.txt").read_text(encoding="utf-8")`.

In [ ]:
# ✍️ passo 5

**Preveja:** o arquivo sai com duas linhas separadas? E se você esquecesse os `\n`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai com as duas linhas. Sem os `\n`, sairia tudo **grudado numa linha só**:
`write` escreve exatamente o texto que recebe.

</details>

**✍️ Passo 6.** Abra o **mesmo** arquivo de novo com `"w"` e escreva só `"3 alarmes criticos\n"`.
Imprima o conteúdo.

In [ ]:
# ✍️ passo 6

**Preveja:** o que tem no arquivo agora — três linhas ou uma?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Uma** linha. O `"w"` apagou tudo **no momento da abertura**, antes de escrever
qualquer coisa. Para acrescentar sem apagar, o modo é `"a"` — troque e rode de novo
para ver a diferença.

</details>

> ⚠️ **Armadilha.** Abrir para escrever esperando "continuar de onde parou". E a pergunta que importa:
> *e se o nome do arquivo de saída for igual ao da entrada?* O `"w"` apaga o arquivo
> que você ia ler — e não há desfazer.

### 🎯 Sua vez — Separar os críticos

Escreva `grava_criticos(entrada, saida)`, que lê o log `entrada`, escreve **só as
linhas críticas** (limpas, uma por linha) no arquivo `saida` e devolve quantas
escreveu.

In [ ]:
def grava_criticos(entrada, saida):
    # sua solução aqui
    pass

In [ ]:
confere(grava_criticos, [
    (("trabalho/alarmes.log", "trabalho/criticos.txt"), 3),
])

<details>
<summary><b>💡 Dica</b></summary>

Dá para abrir os dois arquivos: o de entrada com o modo padrão, o de saída com
`"w"`. Um `with` dentro do outro funciona. Não esqueça o `"\n"` no `write`. Depois,
abra `trabalho/criticos.txt` no painel de arquivos para conferir.

</details>

## 4. CSV

CSV é texto com os campos separados por vírgula e, na primeira linha, o
**cabeçalho** com o nome das colunas. `csv.DictReader` lê uma linha por vez **já
como dicionário**, usando o cabeçalho como chaves. Rode a célula abaixo para ver o
arquivo cru.

📖 [capítulo 7 · CSV](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#csv)

In [ ]:
print(Path("trabalho/medicoes.csv").read_text(encoding="utf-8"))

**✍️ Passo 7.** Escreva `import csv`. Abra `"trabalho/medicoes.csv"` com `encoding="utf-8"` **e**
`newline=""` e percorra `for registro in csv.DictReader(arquivo):` imprimindo
`registro`.

In [ ]:
# ✍️ passo 7

**Preveja:** o arquivo tem 4 linhas. Quantas o laço imprime? O cabeçalho aparece?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Imprime **3** — um dicionário por medição. O cabeçalho **não volta como dado**:
ele virou as chaves de cada dicionário. O `newline=""` é a forma recomendada de
abrir arquivo para o módulo `csv`.

</details>

**✍️ Passo 8.** Troque o `print(registro)` por `print(registro["equipamento"], registro["estado"])`.

In [ ]:
# ✍️ passo 8

**Preveja:** por que não usar `linha.split(",")[0]` em vez disso?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai o nome e o estado de cada linha. O acesso é por **nome de coluna**: se amanhã
o arquivo ganhar uma coluna nova no meio, `registro["estado"]` continua certo, e um
`campos[2]` passaria a pegar outra coisa. E um campo com vírgula dentro, entre
aspas (`"Fulano, Silva"`), o `split(",")` cortaria no meio; o `csv` não.

</details>

## 5. Guardando as linhas para usar depois

O `DictReader` só funciona com o arquivo aberto, e **passa uma vez** pelas linhas.
Para usar os registros depois, acumula-se numa lista — o laço de sempre.

📖 [capítulo 7 · Guardando as linhas para usar depois](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#guardando-as-linhas-para-usar-depois)

**✍️ Passo 9.** Crie `medicoes = []` e, no laço do `DictReader`, faça `medicoes.append(registro)`.
Depois do `with`, imprima `len(medicoes)` e `medicoes[0]["equipamento"]`.

In [ ]:
# ✍️ passo 9

**Preveja:** que tipo de coisa é `medicoes` depois do laço?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`3` e `OLT-CENTRO-01`. `medicoes` é uma **lista de dicionários** — exatamente o
inventário da Aula 06. O laço é o padrão acumulador, juntando registros em vez de
números.

</details>

**✍️ Passo 10.** Agora tente percorrer o leitor **duas vezes**. Dentro de um `with`, faça
`leitor = csv.DictReader(arquivo)`; conte os registros num primeiro `for` (em
`primeira`) e, logo depois, conte de novo num segundo `for` (em `segunda`). Imprima
`primeira, segunda`.

In [ ]:
# ✍️ passo 10

**Preveja:** o segundo laço sobre o mesmo leitor conta quantos registros?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai `3 0`. O leitor **avança e acaba**: depois da primeira passada, não sobra nada
para a segunda. A lista, ao contrário, responde quantas vezes você perguntar — é
por isso que se guarda numa lista.

</details>

## 6. O CSV vira o que você já sabe processar

Com a lista de dicionários em mãos, é a Unidade 2 de novo: acumulador, filtro,
contagem. Com um detalhe que muda tudo: **todo valor que vem do arquivo é texto**.

📖 [capítulo 7 · O CSV vira o que você já sabe processar](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#o-csv-vira-o-que-voce-ja-sabe-processar)

**✍️ Passo 11.** Some as potências: `soma = 0` e, para cada `medicao` em `medicoes`,
`soma = soma + medicao["potencia_dbm"]`. Imprima `soma`.

In [ ]:
# ✍️ passo 11

**Preveja:** por que esta soma falha, se a coluna tem números?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`TypeError`: `medicao["potencia_dbm"]` é o **texto** `"-21.4"`, não o número. O
`csv` não adivinha tipo — tudo vem como `str`. É o mesmo erro da Aula 01, agora com
dado de verdade.

</details>

**✍️ Passo 12.** Corrija com `float(...)` em volta de `medicao["potencia_dbm"]` e imprima a média
`round(soma / len(medicoes), 2)`.

In [ ]:
# ✍️ passo 12

**Preveja:** que número sai?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai `-22.73`. A conversão é **sua** responsabilidade, no ponto em que o valor vai
entrar numa conta. Filtro (quem está `UP`) e contagem (quantos por estado) são os
mesmos padrões de sempre — o capítulo mostra os três lado a lado.

</details>

### 🎯 Sua vez — As potências de quem está no ar

Escreva `potencias_no_ar(caminho)`, que lê o CSV e devolve a lista das potências
(**como números**) dos equipamentos com estado `UP`, na ordem do arquivo.

In [ ]:
import csv


def potencias_no_ar(caminho):
    # sua solução aqui
    pass

In [ ]:
confere(potencias_no_ar, [(("trabalho/medicoes.csv",), [-21.4, -27.0])])

<details>
<summary><b>💡 Dica</b></summary>

Filtro dentro do laço do `DictReader`: **se** `registro["estado"] == "UP"`, faça
`append(float(registro["potencia_dbm"]))`.

</details>

## 7. Quando o arquivo não está lá

📖 [capítulo 7 · Quando o arquivo não está lá](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#quando-o-arquivo-nao-esta-la)

**✍️ Passo 13.** Tente abrir `"coleta_de_ontem.csv"` — que não existe — e imprimir `arquivo.read()`.

In [ ]:
# ✍️ passo 13

**Preveja:** qual é a mensagem quando o caminho está errado?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`FileNotFoundError: [Errno 2] No such file or directory: 'coleta_de_ontem.csv'`. A
mensagem diz **qual caminho** o Python procurou — leia-o: quase sempre o erro é o
caminho, e não o programa.

</details>

> ⚠️ **Armadilha.** No Colab, o caso mais comum é outro: a célula ⚙️ não rodou **nesta sessão**. Os
> arquivos criados somem quando o ambiente reinicia. Na Aula 08, você aprende a
> tratar esse erro em vez de deixar o programa morrer.

## 📟 Resolvendo o chamado

O arquivo tem uma linha por equipamento por hora. Veja o começo dele:

In [ ]:
print(Path("trabalho/estados_do_dia.csv").read_text(encoding="utf-8")[:150])

### 🎯 Sua vez — A disponibilidade

Escreva `disponibilidade(caminho)`, que devolve um dicionário **equipamento →
porcentagem de horas em `UP`**, arredondada com **1 casa**.

In [ ]:
import csv


def disponibilidade(caminho):
    # sua solução aqui
    pass

In [ ]:
confere(disponibilidade, [
    (("trabalho/estados_do_dia.csv",),
     {"OLT-CENTRO-01": 83.3, "ONU-SUL-4512": 66.7, "OLT-NORTE-02": 100.0}),
])

<details>
<summary><b>💡 Dica</b></summary>

Duas contagens da Aula 05 no mesmo laço: `total` (quantas horas cada equipamento
tem no arquivo) e `no_ar` (quantas delas em `UP` — use `get(nome, 0)` para quem
nunca esteve `UP`). No fim, percorra `total.items()` e monte o resultado com
`round(100 * no_ar.get(nome, 0) / horas, 1)`.

</details>

**Resposta ao chamado:** a `ONU-SUL-4512` ficou no ar só 66,7% das horas coletadas.
Se o contrato promete 99%, esse é o enlace que vai gerar crédito para o cliente — e
agora o cálculo roda sozinho, todo dia, sobre o arquivo inteiro.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** Um arquivo com 3 alarmes, gravado normalmente (terminando com quebra de
linha), é percorrido pelo `for` e cada linha é impressa com `repr`. Todas terminam
com:
a) nada  b) `\n`  c) um espaço  d) depende do sistema

<details>
<summary><b>Resposta da 1</b></summary>

**b**. A quebra de linha faz parte da linha lida — por isso o `strip()`.

</details>

**2.** Abrir com `"w"` um arquivo que já existe:
a) acrescenta no fim  b) dá erro  c) apaga o conteúdo anterior  d) cria outro arquivo

<details>
<summary><b>Resposta da 2</b></summary>

**c**, no momento da abertura.

</details>

**3.** `registro["potencia_dbm"] + 1`, com o CSV lido pelo `DictReader`:
a) soma 1 à potência  b) dá `TypeError`  c) concatena `"1"` no fim  d) devolve `None`

<details>
<summary><b>Resposta da 3</b></summary>

**b**. O valor é `str`, e somar `str` com `int` não existe. Faltou o `float()`.

</details>

## 🏠 Para casa

- [Lista 07](https://lacouth.github.io/python_telecom-site/listas/lista07/) —
  arquivos e CSV, com testes automáticos no Colab.
- Releia o [capítulo 7 do site](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/).
- **Na próxima aula:** mini-teste sobre esta aula (linha em branco, `"w"` e
  `DictReader`).